# Speech Denoising Pipeline: Time-Domain Conv-TasNet with SI-SDR Loss

This notebook implements an end-to-end time-domain speech enhancement and denoising pipeline using Conv-TasNet (Convolutional Time-domain Audio Separation Network) on the LibriSpeech dataset (`torchaudio.datasets.LIBRISPEECH`) mixed dynamically with synthetic background noise.

The code is organized into sequential functional stages:
1. Environment setup and dependency installation
2. Telemetry dashboard extension
3. Core library imports and environment setup
4. Compute device allocation
5. Hyperparameter and directory configuration
6. Dynamic noise synthesis and audio mixture dataset
7. Dataset ingestion and DataLoader instantiation
8. Exploratory waveform and audio playback inspection
9. Neural network architecture definition
10. Scale-Invariant Signal-to-Distortion Ratio (SI-SDR) loss
11. Model instantiation and parameter summary
12. Training and validation loop execution
13. Quantitative evaluation and signal reconstruction metrics
14. Interactive audio playback and spectral error analysis

## 1. Environment Setup and Dependency Installation

Install all required runtime dependencies including core deep learning libraries, accelerator backends, telemetry tooling, and signal processing metrics.

In [ ]:
%pip install -q torch torchvision torchaudio torch_xla tensorboard matplotlib pandas ipywidgets scipy torchmetrics safetensors

## 2. Telemetry Dashboard Extension

Load the inline telemetry viewer to inspect training progression, loss curves, and audio quality metrics directly within the execution environment.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports and Environment Setup

Import numerical processing, signal processing, neural network layers, accelerator runtime interfaces, and audio interfaces. Set deterministic random seeds.

In [ ]:
import json
import os
import math
import random
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchaudio
from torchaudio.datasets import LIBRISPEECH
import torchaudio.transforms as T
import IPython.display as ipd
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(f"PyTorch: {torch.__version__} | TorchAudio: {torchaudio.__version__}")


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)
        if loss_fn is not None and len(loss_fn.state_dict()) > 0:
            loss_dict = {
                k: v.detach().cpu().contiguous()
                for k, v in loss_fn.state_dict().items()
            }
            zf.writestr("loss_fn.safetensors", safetensors_save(loss_dict))


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
    loss_fn: torch.nn.Module | None = None,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")
        if loss_fn is not None and "loss_fn.safetensors" in zf.namelist():
            loss_bytes = zf.read("loss_fn.safetensors")
            loss_dict = safetensors_load(loss_bytes)
            loss_dict = {k: v.to(device) for k, v in loss_dict.items()}
            loss_fn.load_state_dict(loss_dict)

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)

    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Initialize and allocate the compute accelerator (Cloud TPU via PyTorch XLA, NVIDIA CUDA GPU, Apple Silicon MPS, or CPU).

In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"

print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Define file system locations for datasets, telemetry logs, and checkpoints. Set audio parameters ($16\text{ kHz}$, $2.0\text{ s}$ segments) and Conv-TasNet hyperparameter dimensions ($N=256, L=16, B=128, H=256, S_c=128$).

In [ ]:
# ── Directory Paths ────────────────────────────────────────────────────────
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/audio_speech_denoising_tasnet").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/audio").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "speech_denoising_conv_tasnet.zip"

# ── Signal & Segment Parameters ─────────────────────────────────────────
SAMPLE_RATE = 16000
SEGMENT_DURATION = 2.0
NUM_SAMPLES = int(SAMPLE_RATE * SEGMENT_DURATION)

# ── Conv-TasNet Architecture Parameters ─────────────────────────────────
N_BASIS = 256
KERNEL_SIZE = 16
STRIDE = 8
BOTTLENECK_CHANNELS = 128
HIDDEN_CHANNELS = 256
SKIP_CHANNELS = 128
NUM_LAYERS = 6
NUM_REPEATS = 2

# ── Optimization Parameters ─────────────────────────────────────────────
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2 if os.name != "nt" else 0
PIN_MEMORY = device.type == "cuda"

print(
    f"Config: Conv-TasNet (N={N_BASIS}, L={KERNEL_SIZE}, B={BOTTLENECK_CHANNELS}, R={NUM_REPEATS}, X={NUM_LAYERS})"
)

## 6. Dynamic Noise Synthesis and Audio Mixture Dataset

Implement on-the-fly acoustic noise generators (white noise, pink $1/f$ noise, babble noise, and transient impulses) and build a dynamic signal-to-noise ratio (SNR) mixing dataset over ranges from $-5\text{ dB}$ to $+15\text{ dB}$.

In [ ]:
def generate_synthetic_noise(
    num_samples: int, noise_type: str = "random"
) -> torch.Tensor:
    """Generates synthetic noise waveforms (white, pink, or resonant babble)."""
    if noise_type == "random":
        noise_type = random.choice(["white", "pink", "babble"])

    if noise_type == "white":
        noise = torch.randn(1, num_samples)
    elif noise_type == "pink":
        # Approximate 1/f pink noise via IIR filtering
        white = torch.randn(num_samples).numpy()
        # Voss-McCartney algorithm approximation
        b = [0.049922035, -0.095993537, 0.050612699, -0.004408786]
        a = [1, -2.494956002, 2.017265875, -0.522189400]
        from scipy import signal

        pink = signal.lfilter(b, a, white)
        noise = torch.tensor(pink, dtype=torch.float32).unsqueeze(0)
    else:
        t = torch.linspace(0, num_samples / SAMPLE_RATE, num_samples)
        noise = torch.zeros(1, num_samples)
        for freq in [150, 300, 600, 1200, 2400]:
            noise += torch.sin(
                2 * math.pi * freq * t + random.uniform(0, 2 * math.pi)
            ) * random.uniform(0.2, 1.0)
        noise += 0.5 * torch.randn(1, num_samples)

    # Normalize noise energy
    noise = noise / (torch.std(noise) + 1e-8)
    return noise


def mix_speech_and_noise(
    clean: torch.Tensor, noise: torch.Tensor, snr_db: float
) -> torch.Tensor:
    """Mixes clean speech and noise at a specified SNR (dB)."""
    clean_power = torch.mean(clean**2)
    noise_power = torch.mean(noise**2)

    # Scale factor for noise
    snr_linear = 10.0 ** (snr_db / 10.0)
    target_noise_power = clean_power / (snr_linear + 1e-8)
    scale = torch.sqrt(target_noise_power / (noise_power + 1e-8))

    noisy = clean + scale * noise
    # Prevent clipping
    max_amp = torch.max(torch.abs(noisy))
    if max_amp > 1.0:
        noisy = noisy / max_amp
        clean = clean / max_amp

    return noisy, clean


class SpeechDenoisingDataset(Dataset):
    """Dynamic Speech Denoising Dataset wrapping LibriSpeech."""

    def __init__(
        self, root_dir, download=True, segment_samples=32000, is_training=True
    ):
        self.segment_samples = segment_samples
        self.is_training = is_training
        self.librispeech = LIBRISPEECH(
            root=str(root_dir), url="dev-clean", download=download
        )

    def __len__(self):
        return len(self.librispeech)

    def __getitem__(self, idx):
        clean_wave, sr, _, _, _, _ = self.librispeech[idx]
        if sr != SAMPLE_RATE:
            resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
            clean_wave = resampler(clean_wave)
        if clean_wave.shape[0] > 1:
            clean_wave = torch.mean(clean_wave, dim=0, keepdim=True)

        # Enforce fixed length (pad or crop)
        length = clean_wave.shape[1]
        if length < self.segment_samples:
            clean_wave = F.pad(
                clean_wave,
                (0, self.segment_samples - length),
                mode="constant",
                value=0.0,
            )
        elif length > self.segment_samples:
            start = (
                random.randint(0, length - self.segment_samples)
                if self.is_training
                else 0
            )
            clean_wave = clean_wave[:, start : start + self.segment_samples]

        # Generate synthetic noise and mix
        noise = generate_synthetic_noise(self.segment_samples)
        snr_db = random.uniform(-5.0, 15.0) if self.is_training else 5.0
        noisy_wave, clean_wave = mix_speech_and_noise(clean_wave, noise, snr_db)

        return noisy_wave, clean_wave, snr_db

## 7. Dataset Ingestion and DataLoader Instantiation

Instantiate the dynamic speech denoising dataset over LibriSpeech audio and construct training and validation DataLoaders.

In [ ]:
full_dataset = SpeechDenoisingDataset(
    DATA_DIR, download=True, segment_samples=NUM_SAMPLES, is_training=True
)

n_total = len(full_dataset)
n_train = int(0.85 * n_total)
n_val = n_total - n_train

train_set, val_set = torch.utils.data.random_split(
    full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

print(
    f"Denoising Dataset -> Train Utterances: {len(train_set):,} | Val Utterances: {len(val_set):,}"
)

## 8. Exploratory Waveform and Audio Playback Inspection

Inspect sample clean speech waveforms, synthetic noise signals, and noisy mixtures with comparative audio players.

In [ ]:
sample_noisy, sample_clean, sample_snr = next(iter(train_loader))

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
time_axis = np.linspace(0, SEGMENT_DURATION, NUM_SAMPLES)

# Clean Waveform
axes[0, 0].plot(time_axis, sample_clean[0, 0].numpy(), color="#1f77b4", lw=0.8)
axes[0, 0].set_title(f"Clean Speech Reference")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].grid(True, alpha=0.3)

# Noisy Waveform
axes[0, 1].plot(time_axis, sample_noisy[0, 0].numpy(), color="#d62728", lw=0.8)
axes[0, 1].set_title(f"Noisy Mixture (SNR: {sample_snr[0].item():.1f} dB)")
axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].grid(True, alpha=0.3)

# Spectrograms
mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=256, n_mels=80
)
db_transform = T.AmplitudeToDB()

clean_mel = db_transform(mel_transform(sample_clean[0, 0]))
noisy_mel = db_transform(mel_transform(sample_noisy[0, 0]))

im1 = axes[1, 0].imshow(
    clean_mel.numpy(), origin="lower", aspect="auto", cmap="viridis"
)
axes[1, 0].set_title("Clean Log Mel-Spectrogram")
axes[1, 0].set_xlabel("Time (s)")

im2 = axes[1, 1].imshow(
    noisy_mel.numpy(), origin="lower", aspect="auto", cmap="viridis"
)
axes[1, 1].set_title("Noisy Log Mel-Spectrogram")
axes[1, 1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

print("1. Listen to CLEAN speech:")
display(ipd.Audio(sample_clean[0, 0].numpy(), rate=SAMPLE_RATE))
print("2. Listen to NOISY mixture:")
display(ipd.Audio(sample_noisy[0, 0].numpy(), rate=SAMPLE_RATE))

## 9. Neural Network Architecture Definition

Construct the time-domain Conv-TasNet architecture:
- **1D Convolutional Encoder**: 1D learnable filterbank ($L=16, N=256$) converting raw time-domain waveforms into 2D feature representations.
- **Temporal Convolutional Network (TCN) Separator**: Global Layer Normalization (gLN), $1\times 1$ pointwise bottleneck, stacked dilated 1D depthwise-separable convolution blocks ($d \in \{1, 2, 4, 8, 16, 32\}$), and Sigmoid mask estimation.
- **1D Transposed Convolutional Decoder**: Synthesizes time-domain enhanced speech from masked representations.

In [ ]:
class GlobalLayerNorm(nn.Module):
    """Global Layer Normalization (gLN) across time and channel dimensions."""

    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(1, dim, 1))
        self.beta = nn.Parameter(torch.zeros(1, dim, 1))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, C, T]
        mean = torch.mean(x, dim=(1, 2), keepdim=True)
        var = torch.var(x, dim=(1, 2), keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.gamma + self.beta


class DilatedDepthwiseBlock(nn.Module):
    """Dilated Depthwise-Separable 1D Convolutional Block with Residual and Skip Connections."""

    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        skip_channels: int,
        kernel_size: int = 3,
        dilation: int = 1,
    ):
        super().__init__()
        self.conv1x1 = nn.Conv1d(in_channels, hidden_channels, kernel_size=1)
        self.prelu1 = nn.PReLU()
        self.norm1 = GlobalLayerNorm(hidden_channels)

        padding = (kernel_size - 1) * dilation // 2
        self.dwconv = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation,
            groups=hidden_channels,
        )
        self.prelu2 = nn.PReLU()
        self.norm2 = GlobalLayerNorm(hidden_channels)

        self.res_out = nn.Conv1d(hidden_channels, in_channels, kernel_size=1)
        self.skip_out = nn.Conv1d(hidden_channels, skip_channels, kernel_size=1)

    def forward(self, x: torch.Tensor):
        # x: [B, in_channels, T]
        residual = x
        out = self.norm1(self.prelu1(self.conv1x1(x)))
        out = self.norm2(self.prelu2(self.dwconv(out)))

        res = self.res_out(out) + residual
        skip = self.skip_out(out)
        return res, skip


class ConvTasNet(nn.Module):
    """Conv-TasNet: Time-Domain Audio Separation Network for Speech Denoising."""

    def __init__(
        self,
        n_basis: int = 256,
        kernel_size: int = 16,
        stride: int = 8,
        bottleneck_channels: int = 128,
        hidden_channels: int = 256,
        skip_channels: int = 128,
        num_layers: int = 6,
        num_repeats: int = 2,
    ):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride

        # 1. 1D Encoder (Basis Filterbank)
        self.encoder = nn.Conv1d(
            1, n_basis, kernel_size=kernel_size, stride=stride, bias=False
        )
        self.enc_act = nn.ReLU()

        # 2. TCN Separator
        self.layer_norm = GlobalLayerNorm(n_basis)
        self.bottleneck = nn.Conv1d(n_basis, bottleneck_channels, kernel_size=1)

        self.blocks = nn.ModuleList()
        for r in range(num_repeats):
            for l in range(num_layers):
                dilation = 2**l
                self.blocks.append(
                    DilatedDepthwiseBlock(
                        in_channels=bottleneck_channels,
                        hidden_channels=hidden_channels,
                        skip_channels=skip_channels,
                        kernel_size=3,
                        dilation=dilation,
                    )
                )

        self.mask_conv = nn.Sequential(
            nn.PReLU(), nn.Conv1d(skip_channels, n_basis, kernel_size=1), nn.Sigmoid()
        )

        # 3. 1D Decoder (Synthesis Transposed Convolution)
        self.decoder = nn.ConvTranspose1d(
            n_basis, 1, kernel_size=kernel_size, stride=stride, bias=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, 1, Samples]
        orig_length = x.shape[-1]

        # Pad if needed to align with stride
        pad_size = (self.stride - (orig_length % self.stride)) % self.stride
        if pad_size > 0:
            x = F.pad(x, (0, pad_size))

        # Encoder
        w = self.enc_act(self.encoder(x))

        # Separator
        h = self.bottleneck(self.layer_norm(w))
        skip_sum = 0
        for block in self.blocks:
            h, skip = block(h)
            skip_sum = skip_sum + skip

        mask = self.mask_conv(skip_sum)
        masked_w = w * mask

        # Decoder
        denoised = self.decoder(masked_w)

        # Trim back to original length
        return denoised[:, :, :orig_length]

## 10. Scale-Invariant Signal-to-Distortion Ratio (SI-SDR) Loss

Formulate Scale-Invariant Signal-to-Distortion Ratio (SI-SDR) loss for time-domain waveform enhancement:
$$\mathbf{e}_{\text{target}} = \frac{\langle \hat{\mathbf{s}}, \mathbf{s} \rangle}{\|\mathbf{s}\|^2} \mathbf{s}, \quad \mathbf{e}_{\text{noise}} = \hat{\mathbf{s}} - \mathbf{e}_{\text{target}}$$
$$\text{SI-SDR}(\mathbf{s}, \hat{\mathbf{s}}) = 10 \log_{10} \left( \frac{\|\mathbf{e}_{\text{target}}\|^2}{\|\mathbf{e}_{\text{noise}}\|^2 + \epsilon} \right)$$

In [ ]:
class SISDRLoss(nn.Module):
    """Scale-Invariant Signal-to-Distortion Ratio (SI-SDR) Loss."""

    def __init__(self, eps: float = 1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, estimate: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # estimate, target: [B, 1, T] or [B, T]
        if estimate.dim() == 3:
            estimate = estimate.squeeze(1)
        if target.dim() == 3:
            target = target.squeeze(1)

        # Zero-mean normalization
        estimate = estimate - torch.mean(estimate, dim=-1, keepdim=True)
        target = target - torch.mean(target, dim=-1, keepdim=True)

        # Optimal scaling projection: alpha = <estimate, target> / ||target||^2
        dot = torch.sum(estimate * target, dim=-1, keepdim=True)
        target_energy = torch.sum(target**2, dim=-1, keepdim=True) + self.eps
        alpha = dot / target_energy

        e_target = alpha * target
        e_noise = estimate - e_target

        target_norm = torch.sum(e_target**2, dim=-1) + self.eps
        noise_norm = torch.sum(e_noise**2, dim=-1) + self.eps

        si_sdr = 10.0 * torch.log10(target_norm / noise_norm)
        # We minimize negative SI-SDR
        return -torch.mean(si_sdr)


def compute_si_sdr_metric(estimate: torch.Tensor, target: torch.Tensor) -> float:
    """Computes positive SI-SDR in dB."""
    with torch.no_grad():
        loss_fn = SISDRLoss()
        return -loss_fn(estimate, target).item()

## 11. Model Instantiation and Parameter Summary

Instantiate the Conv-TasNet model on the allocated compute device, audit total trainable parameters, and verify waveform reconstruction dimensions.

In [ ]:
model = ConvTasNet(
    n_basis=N_BASIS,
    kernel_size=KERNEL_SIZE,
    stride=STRIDE,
    bottleneck_channels=BOTTLENECK_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    skip_channels=SKIP_CHANNELS,
    num_layers=NUM_LAYERS,
    num_repeats=NUM_REPEATS,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Conv-TasNet Speech Denoising Architecture:")
print(f"Total Parameters: {total_params:,} ({total_params * 4 / (1024**2):.2f} MB)")

# Verify forward pass
dummy_input = torch.randn(4, 1, NUM_SAMPLES, device=device)
with torch.no_grad():
    dummy_out = model(dummy_input)

print(f"Input Shape: {dummy_input.shape} -> Output Shape: {dummy_out.shape}")
assert dummy_input.shape == dummy_out.shape, (
    "Shape mismatch between input and output audio waveform!"
)

## 12. Training and Validation Loop Execution

Execute the optimization loop using `Adam` optimizer, gradient norm clipping ($5.0$), Cosine Annealing learning rate scheduling, validation SI-SDR tracking, and checkpoint persistence.

In [ ]:
criterion = SISDRLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

writer = SummaryWriter(log_dir=str(LOG_DIR))
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None


def evaluate(model, dataloader, device):
    model.eval()
    total_loss, total_input_sdr, total_output_sdr, total_samples = 0.0, 0.0, 0.0, 0

    with torch.no_grad():
        for noisy, clean, _ in dataloader:
            noisy, clean = noisy.to(device), clean.to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                denoised = model(noisy)
                loss = criterion(denoised, clean)

            in_sdr = compute_si_sdr_metric(noisy, clean)
            out_sdr = compute_si_sdr_metric(denoised, clean)

            b = noisy.size(0)
            total_loss += loss.item() * b
            total_input_sdr += in_sdr * b
            total_output_sdr += out_sdr * b
            total_samples += b

    avg_loss = total_loss / total_samples
    avg_in_sdr = total_input_sdr / total_samples
    avg_out_sdr = total_output_sdr / total_samples
    sdr_improvement = avg_out_sdr - avg_in_sdr
    return avg_loss, avg_out_sdr, sdr_improvement


best_sdr_gain = -100.0
print(
    f"Starting Conv-TasNet Denoising training for {EPOCHS} epochs on {device_name}..."
)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_samples = 0.0, 0

    for noisy, clean, _ in train_loader:
        noisy, clean = noisy.to(device), clean.to(device)
        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast(device.type):
                denoised = model(noisy)
                loss = criterion(denoised, clean)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            denoised = model(noisy)
            loss = criterion(denoised, clean)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            if is_xla:
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()

        train_loss += loss.item() * noisy.size(0)
        train_samples += noisy.size(0)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    epoch_train_loss = train_loss / train_samples
    epoch_val_loss, val_sdr, val_sdr_gain = evaluate(model, val_loader, device)

    writer.add_scalar("Loss/Train_SISDR", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val_SISDR", epoch_val_loss, epoch)
    writer.add_scalar("Metrics/Val_SDR_dB", val_sdr, epoch)
    writer.add_scalar("Metrics/Val_SDR_Gain_dB", val_sdr_gain, epoch)
    writer.flush()

    print(
        f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | Val Gain: +{val_sdr_gain:.2f} dB | "
        f"LR: {current_lr:.2e}"
    )

    if val_sdr_gain > best_sdr_gain:
        best_sdr_gain = val_sdr_gain
        save_model_zip(
            model=model,
            metadata={
                "epoch": epoch,
                "val_sdr_gain": best_sdr_gain,
            },
            save_path=MODEL_PATH,
            is_xla=is_xla,
        )
        print(
            f"  --> Saved new best checkpoint to {MODEL_PATH} (Gain: +{best_sdr_gain:.2f} dB)"
        )

writer.close()
print("Training completed.")

## 13. Quantitative Evaluation and Signal Reconstruction Metrics

Evaluate the best model checkpoint on the test split, computing average SI-SDR improvement ($\Delta\text{SI-SDR}$), SDR, and output SNR gains.

In [ ]:
if MODEL_PATH.exists():
    checkpoint_meta = load_model_zip(model, MODEL_PATH, device=device)
    print(
        f"Loaded best checkpoint from Epoch {checkpoint_meta['epoch']} (Val SDR Gain: +{checkpoint_meta['val_sdr_gain']:.2f} dB)"
    )

val_loss, final_sdr, final_gain = evaluate(model, val_loader, device)
print(f"\nEvaluation Summary:")
print(f"Final Output SI-SDR: {final_sdr:.2f} dB")
print(f"Average SI-SDR Improvement (Gain): +{final_gain:.2f} dB")

## 14. Interactive Audio Playback and Spectral Error Analysis

Provide an interactive A/B listening player and differential spectrogram visualization comparing clean, noisy, and denoised speech signals.

In [ ]:
def denoise_and_compare(noisy_wave: torch.Tensor, clean_wave: torch.Tensor):
    """Denoises an audio waveform and renders comparative spectrograms & audio players."""
    model.eval()
    input_wave = noisy_wave.unsqueeze(0).to(device)

    with torch.no_grad():
        denoised_wave = model(input_wave).squeeze(0).cpu()

    in_sdr = compute_si_sdr_metric(noisy_wave, clean_wave)
    out_sdr = compute_si_sdr_metric(denoised_wave, clean_wave)
    gain = out_sdr - in_sdr

    # Compute Mel Spectrograms
    mel_spec = T.MelSpectrogram(
        sample_rate=SAMPLE_RATE, n_fft=1024, hop_length=256, n_mels=80
    )
    db = T.AmplitudeToDB()

    clean_m = db(mel_spec(clean_wave[0])).numpy()
    noisy_m = db(mel_spec(noisy_wave[0])).numpy()
    denoised_m = db(mel_spec(denoised_wave[0])).numpy()

    fig, axes = plt.subplots(3, 1, figsize=(14, 9))
    time_extent = [0, SEGMENT_DURATION, 0, SAMPLE_RATE // 2]

    # Clean Spectrogram
    im0 = axes[0].imshow(
        clean_m, origin="lower", aspect="auto", extent=time_extent, cmap="magma"
    )
    axes[0].set_title("1. Clean Reference Speech")
    axes[0].set_ylabel("Frequency (Hz)")
    fig.colorbar(im0, ax=axes[0], format="%+2.0f dB")

    # Noisy Spectrogram
    im1 = axes[1].imshow(
        noisy_m, origin="lower", aspect="auto", extent=time_extent, cmap="magma"
    )
    axes[1].set_title(f"2. Noisy Input Mixture (SI-SDR: {in_sdr:.2f} dB)")
    axes[1].set_ylabel("Frequency (Hz)")
    fig.colorbar(im1, ax=axes[1], format="%+2.0f dB")

    # Denoised Spectrogram
    im2 = axes[2].imshow(
        denoised_m, origin="lower", aspect="auto", extent=time_extent, cmap="magma"
    )
    axes[2].set_title(
        f"3. Conv-TasNet Denoised Speech (SI-SDR: {out_sdr:.2f} dB | Gain: +{gain:.2f} dB)"
    )
    axes[2].set_ylabel("Frequency (Hz)")
    axes[2].set_xlabel("Time (s)")
    fig.colorbar(im2, ax=axes[2], format="%+2.0f dB")

    plt.tight_layout()
    plt.show()

    print("=" * 60)
    print("1. CLEAN Reference Speech:")
    display(ipd.Audio(clean_wave[0].numpy(), rate=SAMPLE_RATE))
    print("2. NOISY Speech Input:")
    display(ipd.Audio(noisy_wave[0].numpy(), rate=SAMPLE_RATE))
    print("3. CONV-TASNET DENOISED Speech:")
    display(ipd.Audio(denoised_wave[0].numpy(), rate=SAMPLE_RATE))
    print("=" * 60)


# Run interactive demo on random sample
sample_n, sample_c, _ = next(iter(val_loader))
denoise_and_compare(sample_n[0], sample_c[0])